<a href="https://colab.research.google.com/github/JakeReas/21_days_Anirudha_Kulkarni/blob/main/ADS_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Reading LIbraries

In [ ]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

### Reading Docs

In [ ]:
df_dx = pd.read_parquet('dx_data.parquet')
df_px = pd.read_parquet('px_data.parquet')
df_rx = pd.read_parquet('rx_data.parquet')
df_dx_map = pd.read_csv('dx_mapping.csv')
df_px_map = pd.read_csv('px_mapping.csv')
df_rx_map = pd.read_excel('rx_map.xlsx')

### EDA

##### Creating a unified document by merging

In [ ]:
df_dx_ov = pd.merge(df_dx, df_dx_map, left_on = 'claim_diagnoses_diagnosis_cd', right_on = 'dx_code', how = 'inner')
df_dx_ov['claim_code'] = df_dx_ov['claim_diagnoses_diagnosis_cd']
df_dx_ov['event_name'] = df_dx_ov['category_1']+'_'+df_dx_ov['category_2']
df_dx_ov.head()

,patient_id,claim_nbr,claim_date,provider_id,claim_category,payer_1_id,primary_hco,claim_diagnoses_diagnosis_cd,dx_code,description,category_1,category_2,claim_code,event_name
0,xb7359516,2b628eec,2023-06-09,1853858588,THERAPY (SVC),50485,1.073587e+09,R41841,R41841,Cognitive_communication_deficit,Cognitive_Impairment,Inclusion,R41841,Cognitive_Impairment_Inclusion
1,xd59d1581,21139594,2024-09-13,1822544979,OFFICE (SVC),53150,1.689608e+09,F0280,F0280,Dementia_in_other_diseases_classified_elsewher...,Dementia,Inclusion,F0280,Dementia_Inclusion
2,x1b325151,129d51b7,2023-09-28,1823534981,OFFICE (SVC),53150,1.790391e+09,R413,R413,Other_amnesia,Memory_Loss,Inclusion,R413,Memory_Loss_Inclusion
3,xc3b46c54,cddc141c,2023-03-14,1875060577,OFFICE (SVC),84934,1.053555e+09,R419,R419,Unspecified_Symptoms_And_Signs_Involving_Cogni...,Cognitive_Impairment,Inclusion,R419,Cognitive_Impairment_Inclusion
4,x5e18d525,188761c9,2022-05-27,1812413053,INPATIENT (SVC),48293,1.407878e+09,F0391,F0391,Unspecified_dementia_with_behavioral_disturbance,Dementia,Inclusion,F0391,Dementia_Inclusion


In [ ]:
df_px_ov = pd.merge(df_px, df_px_map, left_on = 'procedure_cd', right_on = 'px_code', how = 'inner')
df_px_ov['claim_code'] = df_px_ov['procedure_cd']
df_px_ov['event_name'] = df_px_ov['category_1']+'_'+df_px_ov['category_2']
df_px_ov.head()

,patient_id,claim_nbr,claim_date,provider_id,claim_category,payer_1_id,primary_hco,procedure_cd,px_code,Description,category_1,category_2,claim_code,event_name
0,x91331251,16714971,2022-08-12,1805340847,HOSPICE (SVC),53150,1.386051e+09,Q5009,Q5009,HOSPICE_OR_HOME_HEALTH_CARE_PROVIDED_IN_PLACE_NO,Hospice_&_Long_Term_Care,Severe_AD_Patients,Q5009,Hospice_&_Long_Term_Care_Severe_AD_Patients
1,x76839e12,21918182,2022-07-12,1870089508,HOME HEALTH (SVC),53150,1.316934e+09,Q5001,Q5001,HOSPICE_OR_HOME_HEALTH_CARE_PROVIDED_IN_PATIENT',Hospice_&_Long_Term_Care,Severe_AD_Patients,Q5001,Hospice_&_Long_Term_Care_Severe_AD_Patients
2,xc52e79c1,2211352e,2023-05-16,1829582782,LAB (SVC),53150,1.649260e+09,96365,96365,IV_Infusion_Therapy/Prophylaxis_/Dx_1St_To_1_Hr,Intravenous_infusion,NaN,96365,NaN
3,xe61e71e2,21414442,2024-11-01,1827564356,HOME HEALTH (SVC),-1,1.750898e+09,Q5001,Q5001,HOSPICE_OR_HOME_HEALTH_CARE_PROVIDED_IN_PATIENT',Hospice_&_Long_Term_Care,Severe_AD_Patients,Q5001,Hospice_&_Long_Term_Care_Severe_AD_Patients
4,x3e131112,1912e1c6,2022-10-05,1826585952,INPATIENT (POS),50909,1.407840e+09,96365,96365,IV_Infusion_Therapy/Prophylaxis_/Dx_1St_To_1_Hr,Intravenous_infusion,NaN,96365,NaN


In [ ]:
df_rx_ov = pd.merge(df_rx, df_rx_map, left_on = 'ndc', right_on = 'NDC', how = 'inner')
df_rx_ov['claim_code' ] = df_rx_ov['ndc']
df_rx_ov['claim_nbr'] = df_rx_ov['rx_claim_nbr']
df_rx_ov['event_name'] = df_rx_ov['Generic Name']
df_rx_ov.head()

,patient_id,rx_claim_nbr,claim_date,provider_id,ndc,payer_id,claim_status,patient_to_pay_amt,NDC,Generic Name,claim_code,claim_nbr,event_name
0,x26999786,11621194,2023-12-22,1879004053,abcd,3439,Dispensed,19.45,abcd,STAR_DRUG,abcd,11621194,STAR_DRUG
1,x26999786,11621194,2023-12-22,1879004053,abcd,3439,Dispensed,19.45,abcd,STAR_DRUG,abcd,11621194,STAR_DRUG
2,x26999786,11621194,2023-12-22,1879004053,abcd,3439,Dispensed,19.45,abcd,STAR_DRUG,abcd,11621194,STAR_DRUG
3,x26999786,11621194,2023-12-22,1879004053,abcd,3439,Dispensed,19.45,abcd,STAR_DRUG,abcd,11621194,STAR_DRUG
4,xd61c78c4,c99918c2,2023-11-28,1837685377,29300017105,85866,Dispensed,0.00,29300017105,MEMANTINE_HCL,29300017105,c99918c2,MEMANTINE_HCL


In [ ]:
# Concatenate the DataFrames
df_ov = pd.concat([df_dx_ov[['patient_id', 'claim_nbr', 'claim_date', 'provider_id', 'claim_code','event_name']],
                   df_px_ov[['patient_id', 'claim_nbr', 'claim_date', 'provider_id', 'claim_code','event_name']],
                   df_rx_ov[['patient_id', 'claim_nbr', 'claim_date', 'provider_id', 'claim_code','event_name']]])

# Display the first few rows of the concatenated DataFrame
df_ov.head()


,patient_id,claim_nbr,claim_date,provider_id,claim_code,event_name
0,xb7359516,2b628eec,2023-06-09,1853858588,R41841,Cognitive_Impairment_Inclusion
1,xd59d1581,21139594,2024-09-13,1822544979,F0280,Dementia_Inclusion
2,x1b325151,129d51b7,2023-09-28,1823534981,R413,Memory_Loss_Inclusion
3,xc3b46c54,cddc141c,2023-03-14,1875060577,R419,Cognitive_Impairment_Inclusion
4,x5e18d525,188761c9,2022-05-27,1812413053,F0391,Dementia_Inclusion


In [ ]:
print(df_ov[df_ov['event_name']=='STAR_DRUG']['patient_id'].nunique())
df_ov['OUTCOME_FLAG'] = df_ov['event_name'].apply(lambda x: 1 if x == 'STAR_DRUG' else 0)
df_ov.tail()

1958


,patient_id,claim_nbr,claim_date,provider_id,claim_code,event_name,OUTCOME_FLAG
1784798,x991e2427,16b164d2,2024-03-23,1874057150,43547027611,DONEPEZIL_HCL,0
1784799,x8de54252,88b1c2b9,2022-05-10,1818497480,43547027509,DONEPEZIL_HCL,0
1784800,x2637c79e,81dd611c,2023-11-25,1867970287,43547027609,DONEPEZIL_HCL,0
1784801,xd861d29c,c7125249,2022-04-04,1866900341,43547027511,DONEPEZIL_HCL,0
1784802,xdc219484,17418d52,2023-08-10,1828504139,72603011902,MEMANTINE_HCL,0


#### Feature or event selection based on user input threshold

In [ ]:
total_unique_patients = df_ov['patient_id'].nunique()
event_reach = df_ov.groupby('event_name')['patient_id'].nunique()/total_unique_patients
threshold = input('What is the threshold for the event reach?: ')
filtered_events = event_reach[event_reach > float(threshold)]
filtered_events

What is the threshold for the event reach?: 0.05


,patient_id
event_name,
AD_Inclusion,0.171012
Cognitive_Impairment_Inclusion,0.341052
DONEPEZIL_HCL,0.210025
Dementia_Inclusion,0.186202
MEMANTINE_HCL,0.163334
Memory_Loss_Inclusion,0.203623


In [ ]:
filtered_df_ov = df_ov[df_ov['event_name'].isin(filtered_events.index)]
print('shape of filtered df from ',df_ov.shape, 'to ', filtered_df_ov.shape)

shape of filtered df from  (4730559, 7) to  (3347864, 7)


### Creating features - Binary and Frequency Features

In [ ]:
def create_binary_features(list_of_values, cat_col, filtered_df_ov):

    if not list_of_values:
        list_of_values = filtered_df_ov[cat_col].unique().tolist()

    # Create a pivot table with binary values
    binary_feats_df = filtered_df_ov.pivot_table(
        index='patient_id',
        columns=cat_col,
        values='claim_date',
        aggfunc=lambda x: 1,
        fill_value=0
    ).reset_index()

    # Rename columns to include '_binary'
    binary_feats_df.columns = [f'{col}_binary' if col != 'patient_id' else col for col in binary_feats_df.columns]
    df_ov_bin = pd.merge(df_ov, binary_feats_df, on = 'patient_id', how = 'left')
    return df_ov_bin
list_of_values = []
df_ov_bin = create_binary_features(list_of_values, cat_col = 'event_name',filtered_df_ov = filtered_df_ov)

In [ ]:
def create_frequency_features(list_of_values, time_frame_list, cat_col, filtered_df_ov):
    if not list_of_values:
        list_of_values = filtered_df_ov[cat_col].unique().tolist()

    # Ensure 'claim_date' is in datetime format
    filtered_df_ov['claim_date'] = pd.to_datetime(filtered_df_ov['claim_date'])

    # Create a DataFrame to store the frequency features
    frequency_feats_df = pd.DataFrame()

    for time_frame in time_frame_list:
        for event in list_of_values:
            event_col_name = f'{event}_{time_frame}_freq'
            filtered_df_ov[event_col_name] = filtered_df_ov.groupby('patient_id').apply(
                lambda group: group.assign(
                    **{event_col_name: group.apply(
                        lambda row: group[(group[cat_col] == event) &
                                          (group['claim_date'] >= row['claim_date'] - pd.Timedelta(time_frame)) &
                                          (group['claim_date'] <= row['claim_date'])].shape[0], axis=1)
                    }
                )
            )[event_col_name].reset_index(level=0, drop=True)
            print(event_col_name)
        print(time_frame)

    # Aggregate the frequency features at the patient level
    frequency_feats_df = filtered_df_ov.groupby('patient_id').agg(
        {f'{event}_{time_frame}_freq': 'sum' for time_frame in time_frame_list for event in list_of_values}
    ).reset_index()

    frequency_feats_df = pd.merge(filtered_df_ov[['patient_id']].drop_duplicates(), frequency_feats_df, on='patient_id', how='left')

    return frequency_feats_df

# Example usage
list_of_values = []
df_ov_bin = create_frequency_features(list_of_values=list_of_values, time_frame_list=['30d', '60d', '90d'], cat_col='event_name', filtered_df_ov=df_ov_bin)
df_ov_freq.head()


In [ ]:
df_rx = pd.read_excel('rx_map.xlsx')
df_rx['Generic Name'].value_counts()

## Features extracted using LLM

## Hyperparamater Tuning

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from tqdm import tqdm, tqdm_notebook
import pickle
# import pyarrow
import time
# from joblib import Parallel, delayed
from math import sqrt
import warnings
import gc
warnings.filterwarnings("ignore")
from sklearn import datasets
import os
from numpy import loadtxt
!pip install optuna
import optuna
from optuna import Trial, visualization
from optuna.samplers import TPESampler
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import confusion_matrix
from sklearn.metrics import average_precision_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import plot_tree,export_text
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
!pip install optuna-integration[lightgbm]
from optuna.integration import LightGBMPruningCallback
from matplotlib import pyplot
import random
import gc
from pathlib import Path
from numpy import arange
import matplotlib.pyplot as plt
import shap
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest, SelectFromModel, chi2
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
from sklearn.metrics import cohen_kappa_score, roc_auc_score, f1_score,precision_recall_curve, precision_recall_fscore_support, auc,  precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from inspect import signature
from lightgbm import LGBMClassifier
import gc
from copy import deepcopy
import copy
gc.collect()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 1.9 MB/s eta 0:00:00


0

In [ ]:
#df_ov_bin['patient_id'] = df_ov_bin['patient_id'].astype(int)
df_ov_bin['provider_id'] = df_ov_bin['provider_id'].astype(int)
#df_ov_bin['patient_id'] = df_ov_bin['patient_id'].astype(int)
df_ov_bin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4730559 entries, 0 to 4730558
Data columns (total 13 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   patient_id                             object 
 1   claim_nbr                              object 
 2   claim_date                             object 
 3   provider_id                            int64  
 4   claim_code                             object 
 5   event_name                             object 
 6   OUTCOME_FLAG                           int64  
 7   AD_Inclusion_binary                    float64
 8   Cognitive_Impairment_Inclusion_binary  float64
 9   DONEPEZIL_HCL_binary                   float64
 10  Dementia_Inclusion_binary              float64
 11  MEMANTINE_HCL_binary                   float64
 12  Memory_Loss_Inclusion_binary           float64
dtypes: float64(6), int64(2), object(5)
memory usage: 469.2+ MB


In [ ]:
X = df_ov_bin.drop(columns = ['OUTCOME_FLAG', 'claim_nbr','claim_code','event_name', 'claim_date','patient_id'])
y = df_ov_bin['OUTCOME_FLAG']

X_train, X_valid, y_train, y_valid = train_test_split(X,y,stratify = y, test_size = 0.2, random_state = 42)
#

In [ ]:
#Optuna objective function
def objective(trial):
  param = {
      'objective': 'binary',
      'metric': 'auc',
      'boosting_type': 'gbdt',
      'verbosity': '-1',
      'learning_rate': trial.suggest_loguniform('learning_rate',1e-3,1e-1),
      'num_leaves': trial.suggest_int('num_leaves', 20, 1000),
      'max_depth': trial.suggest_int('max_depth', 3, 16),
      'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
      'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
      'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
      'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
      'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0)
  }

  dtrain = lgb.Dataset(X_train, label = y_train)
  dvalid = lgb.Dataset(X_valid, label = y_valid)

  model = lgb.train(
    param,
    dtrain,
    valid_sets = [dtrain, dvalid],
    num_boost_round = 1000,
    #early_stopping_rounds = 100,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]  # Log nothing (0), or use 100 if you want occasional logs
)

  preds = model.predict(X_valid)
  auc = roc_auc_score(y_valid, preds)
  return auc


study = optuna.create_study(direction = 'maximize')
study.optimize(objective, n_trials = 50)

print("Best Trial: ")
trial = study.best_trail
print(f" AUC : {trial.value}")
print("Best Params: ")
for key, value in trial.params.items():
  print(f" {key} : {value}")

[I 2025-05-30 07:41:29,807] A new study created in memory with name: no-name-91b54024-35ca-40cf-8b2d-bd9dc0dd7ef5


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1000]	training's auc: 0.877282	valid_1's auc: 0.854796


[I 2025-05-30 08:04:47,785] Trial 0 finished with value: 0.854795778500332 and parameters: {'learning_rate': 0.005040782365132158, 'num_leaves': 747, 'max_depth': 8, 'min_child_samples': 12, 'subsample': 0.6077345010881782, 'colsample_bytree': 0.9033014602499438, 'reg_alpha': 0.0014894438572300192, 'reg_lambda': 0.012404846106615769}. Best is trial 0 with value: 0.854795778500332.


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1000]	training's auc: 0.795635	valid_1's auc: 0.791762


[I 2025-05-30 08:25:00,401] Trial 1 finished with value: 0.7917622140393644 and parameters: {'learning_rate': 0.0017195850332002494, 'num_leaves': 485, 'max_depth': 7, 'min_child_samples': 97, 'subsample': 0.9593489503379498, 'colsample_bytree': 0.8645965145806647, 'reg_alpha': 4.908596488499851, 'reg_lambda': 0.07124253368063507}. Best is trial 0 with value: 0.854795778500332.


...

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[999]	training's auc: 0.838605	valid_1's auc: 0.822551


[I 2025-05-30 09:35:14,324] Trial 5 finished with value: 0.8225508001962352 and parameters: {'learning_rate': 0.03371943328462526, 'num_leaves': 820, 'max_depth': 6, 'min_child_samples': 26, 'subsample': 0.5225370762111149, 'colsample_bytree': 0.5891138315946813, 'reg_alpha': 0.012842282219150753, 'reg_lambda': 6.918741913875839}. Best is trial 2 with value: 0.9159331860165805.


Training until validation scores don't improve for 100 rounds


[W 2025-05-30 09:41:59,204] Trial 6 failed with parameters: {'learning_rate': 0.07994597811138418, 'num_leaves': 718, 'max_depth': 8, 'min_child_samples': 48, 'subsample': 0.921567047679308, 'colsample_bytree': 0.9931894692249124, 'reg_alpha': 1.6134576664809925, 'reg_lambda': 1.3529066849045517} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-39-da2d05d4f2ae>", line 21, in objective
    model = lgb.train(
            ^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/lightgbm/engine.py", line 307, in train
    booster.update(fobj=fobj)
  File "/usr/local/lib/python3.11/dist-packages/lightgbm/basic.py", line 4136, in update
    _LIB.LGBM_BoosterUpdateOneIter(
KeyboardInterrupt
[W 2025-05-30 09:41:59,211] Trial 6 failed with value None.


KeyboardInterrupt: 

## Modeling

In [ ]:
import lightgbm as lgb
import pandas as pd
df_ov_bin_1 = df_ov_bin.drop(columns = ['claim_nbr','claim_code','event_name', 'claim_date','patient_id'])
# 1. Define the Target Variable
X = df_ov_bin_1.drop(columns=['OUTCOME_FLAG','provider_id'])
y = df_ov_bin_1['OUTCOME_FLAG']  # 1 = positive, 0 = unlabeled

# 2. LightGBM dataset
dtrain = lgb.Dataset(X, label=y)

# 3. Best hyperparameters from Optuna (example — replace with your actual ones)
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'learning_rate': 0.01,
    'num_leaves': 64,
    'max_depth': 8,
    'min_child_samples': 20,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1
}

# 4. Train LightGBM for 30–40 iterations (rounds)
pu_model = lgb.train(
    best_params,
    dtrain,
    num_boost_round=35  # You can adjust this between 30–40
)

# 5. Predict on training data or new data
y_scores = pu_model.predict(X)  # Gives likelihood of being positive (between 0 and 1)

# Optional: Check top predicted positives
df_ov_bin_1['PU_Score'] = y_scores
df_ov_bin_sorted = df_ov_bin_1.sort_values('PU_Score', ascending=False)


In [ ]:
df_ov_bin_sorted.head(10)

,provider_id,OUTCOME_FLAG,AD_Inclusion_binary,Cognitive_Impairment_Inclusion_binary,DONEPEZIL_HCL_binary,Dementia_Inclusion_binary,MEMANTINE_HCL_binary,Memory_Loss_Inclusion_binary,PU_Score
981248,1873041464,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
981250,1833646698,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
981297,1869214536,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
980933,1804356575,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
981425,1839617060,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
4730451,1829570057,1,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
47,1805340847,0,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
4730452,1829570057,1,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
4730453,1829570057,1,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
4730454,1829570057,1,NaN,NaN,NaN,NaN,NaN,NaN,0.003619
